In [56]:
import torch
import torchvision
from pathlib import Path
from torchvision.datasets import ImageFolder
print("FILE IMPORTED")

FILE IMPORTED


In [57]:
dataset=Path("dataset")
train_path=dataset/"train"
test_path=dataset/"test"
val_path=dataset/"valid"
print("PATH LOADED")
print(train_path)

PATH LOADED
dataset\train


In [58]:
from torchvision.transforms import transforms
transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
train_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
print("DATASET TRANSFORMATION")

DATASET TRANSFORMATION


In [59]:
train_data=ImageFolder(train_path,
                       transform=train_transform)
test_data=ImageFolder(test_path,
                      transform=transform)
val_data=ImageFolder(val_path,
                     transform=transform)
print("IMAGES INTO IMAGEFOLDER")

IMAGES INTO IMAGEFOLDER


In [60]:
from torch.utils.data import DataLoader
train_loader=DataLoader(train_data,
                        batch_size=32,
                        shuffle=True)
test_loader=DataLoader(test_data,
                        batch_size=32,
                        shuffle=False)
val_loader=DataLoader(val_data,
                        batch_size=32,
                        shuffle=False)
print("DATA LOADER LOADED")

DATA LOADER LOADED


In [61]:
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights
weight=ResNet50_Weights.DEFAULT
model=resnet50(weights=weight)
model.fc=nn.Linear(2048,2)
criterion=nn.CrossEntropyLoss()
print("MODEL LOADED")

MODEL LOADED


In [62]:
for parameter in model.parameters():
    parameter.requires_grad=False
for parameter in model.fc.parameters():
    parameter.requires_grad=True
    
optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr=0.001)
print("OPTIMIZER SET")

OPTIMIZER SET


In [70]:
import copy
best_loss=float("inf")
best_model=None
num=25
for epoch in range(num):
    train_loss=0
    model.train()
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        output=model(inputs)
        loss=criterion(output, labels)
        loss.backward()
        optimizer.step()
        train_loss+=loss.item()
    total_train_loss=train_loss/len(train_loader)
    print("----------------------")
    print(f"TRAIN LOSS:{total_train_loss}")
    model.eval()
    with torch.no_grad():
        val_loss=0
        for inputs, labels in val_loader:
            output=model(inputs)
            loss=criterion(output, labels)
            val_loss+=loss.item()
        total_val_loss=val_loss/len(val_loader)
        if total_val_loss < best_loss:
            best_loss=total_val_loss
            best_model=copy.deepcopy(model.state_dict())
        print(f"VALIDATION LOSS: {total_val_loss}, LEAST VALIDATION LOSS:{best_loss}")
        
    

----------------------
TRAIN LOSS:0.3118140995502472
VALIDATION LOSS: 0.31249749660491943, LEAST VALIDATION LOSS:0.31249749660491943
----------------------
TRAIN LOSS:0.2729317545890808
VALIDATION LOSS: 0.3100743889808655, LEAST VALIDATION LOSS:0.3100743889808655
----------------------
TRAIN LOSS:0.31655891239643097
VALIDATION LOSS: 0.2980540692806244, LEAST VALIDATION LOSS:0.2980540692806244
----------------------
TRAIN LOSS:0.25606849789619446
VALIDATION LOSS: 0.28474751114845276, LEAST VALIDATION LOSS:0.28474751114845276
----------------------
TRAIN LOSS:0.36405184119939804
VALIDATION LOSS: 0.26760226488113403, LEAST VALIDATION LOSS:0.26760226488113403
----------------------
TRAIN LOSS:0.23395311832427979
VALIDATION LOSS: 0.2598254978656769, LEAST VALIDATION LOSS:0.2598254978656769
----------------------
TRAIN LOSS:0.1869967207312584
VALIDATION LOSS: 0.24925930798053741, LEAST VALIDATION LOSS:0.24925930798053741
----------------------
TRAIN LOSS:0.19425998628139496
VALIDATION LOSS: 

In [71]:
torch.save(best_model, "rosemodel.pth")

In [78]:
from PIL import Image
import torch

image_path = "manual/images (1).jpg"

image = Image.open(image_path).convert("RGB")

image = transform(image)

image = image.unsqueeze(0)

model.eval()

with torch.no_grad():
    output = model(image)

    probabilities = torch.softmax(output, dim=1)

    confidence, prediction = torch.max(probabilities, dim=1)

class_names = [
    "RED",
    "WHITE"
]

print("Prediction:", class_names[prediction.item()])
print("Confidence:", confidence.item() * 100, "%")

Prediction: RED
Confidence: 94.28206086158752 %


In [80]:
#MODEL EVALUATION
import torch
from torchmetrics.classification import (
    MulticlassAccuracy,
    MulticlassConfusionMatrix,
    MulticlassPrecision,
    MulticlassRecall,
    MulticlassF1Score
)

model.eval()

# Metrics
accuracy = MulticlassAccuracy(num_classes=2)
precision = MulticlassPrecision(num_classes=2, average="macro")
recall = MulticlassRecall(num_classes=2, average="macro")
f1 = MulticlassF1Score(num_classes=2, average="macro")
confmat = MulticlassConfusionMatrix(num_classes=2)

with torch.no_grad():

    for inputs, labels in test_loader:

        output = model(inputs)

        accuracy.update(output, labels)
        precision.update(output, labels)
        recall.update(output, labels)
        f1.update(output, labels)
        confmat.update(output, labels)

# Results
print("TEST RESULTS")
print("----------------------")
print("Accuracy :", accuracy.compute().item())
print("Precision:", precision.compute().item())
print("Recall   :", recall.compute().item())
print("F1 Score :", f1.compute().item())

print("\nConfusion Matrix:")
print(confmat.compute())

TEST RESULTS
----------------------
Accuracy : 0.9285714626312256
Precision: 0.9375
Recall   : 0.9285714626312256
F1 Score : 0.928205132484436

Confusion Matrix:
tensor([[7, 0],
        [1, 6]])
